# Chakravyuh — Analyzer GRPO Training (Colab)

Minimum-requirement training notebook for the **Meta PyTorch OpenEnv Hackathon 2026**.

This Colab trains a LoRA adapter on **Qwen2.5-3B-Instruct** (fits in free T4) for the Chakravyuh Behavioral Analyzer. A smoke-run (50 episodes, ~90 min on free T4) produces a checkpoint you can drop back into `server/demo_ui.py` for live inference.

- Repo: https://github.com/UjjwalPardeshi/Chakravyuh
- Benchmark: `chakravyuh-bench-v0` (n=135) — 5 fraud categories + benign + novel post-2024
- Paper results: scripted baseline catches **80%** of known scams, **50%** of novel post-2024 attacks (30pp gap, p=0.003)

**Runtime**: select `Runtime → Change runtime type → T4 GPU` before running.

## 1. Setup

In [ ]:
# Install dependencies (takes ~3 min on Colab)
!pip install -q --upgrade "torch>=2.5,<2.6" "transformers>=4.45" "trl>=0.12" \
    "peft>=0.13" "accelerate>=0.33" "bitsandbytes>=0.44" "datasets>=2.19"
!pip install -q --upgrade "unsloth @ git+https://github.com/unslothai/unsloth.git" || echo 'unsloth optional'
!pip install -q "pydantic>=2.6" "numpy>=1.26" "sentence-transformers>=2.7" "wandb"

In [ ]:
# Clone the repo
import os
REPO_URL = 'https://github.com/UjjwalPardeshi/Chakravyuh.git'
if not os.path.exists('Chakravyuh'):
    !git clone $REPO_URL
os.chdir('Chakravyuh')
!pip install -q -e . ; echo 'installed'

In [ ]:
# Confirm GPU + package versions
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
import trl, peft, transformers
print('trl:', trl.__version__)
print('peft:', peft.__version__)
print('transformers:', transformers.__version__)

## 2. Smoke-run baseline (verify env + reward)

In [ ]:
# Build the training dataset + sanity-check reward math (no model loaded)
from training.grpo_analyzer import build_training_examples, compute_reward

examples = build_training_examples()
print(f'Built {len(examples)} examples ({sum(1 for e in examples if e.is_scam)} scam)')

# Sanity: reward on a perfect output vs. wrong output
ex = examples[0]
good = '{"score": 0.95, "signals": ["urgency", "info_request"], "explanation": "OTP + urgency = scam"}'
bad = '{"score": 0.1, "signals": [], "explanation": "Looks fine to me"}'
print('GOOD completion reward:', compute_reward(good, ex).total)
print('BAD  completion reward:', compute_reward(bad, ex).total)

## 3. Train on a small model (50 episodes, ~90 min on T4)

In [ ]:
# NOTE: On free T4 (16 GB VRAM) we use Qwen2.5-3B. On A100/H100 you can swap to Qwen2.5-7B.
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
EPISODES = 50           # smoke run — increase to 200 on A100
OUTPUT_DIR = 'checkpoints/analyzer_lora_colab'

!python -m training.grpo_analyzer \
    --model $MODEL_NAME \
    --episodes $EPISODES \
    --output $OUTPUT_DIR \
    --lora-rank 16 \
    --batch-size 2 \
    --num-generations 4 \
    --no-wandb

## 4. Evaluate the trained adapter against Mode C

In [ ]:
# Use the trained LoRA inside the Mode C runner
from chakravyuh_env.agents.llm_analyzer import LLMAnalyzer
from eval.mode_c_real_cases import load_dataset, run_eval, aggregate, DEFAULT_DATASET

analyzer = LLMAnalyzer(
    model_name=MODEL_NAME,
    lora_path=OUTPUT_DIR,
    use_unsloth=True,
    load_in_4bit=True,
)
analyzer.load()

data = load_dataset(DEFAULT_DATASET)
results = run_eval(analyzer, data, threshold=0.5)
metrics = aggregate(results)
print(f'Detection: {metrics.detection_rate:.1%}')
print(f'Precision: {metrics.precision:.1%}')
print(f'F1: {metrics.f1:.3f}')

## 5. Save the adapter to Google Drive (optional)

In [ ]:
# Mount Drive and copy the adapter for persistence
from google.colab import drive  # type: ignore[import-not-found]
drive.mount('/content/drive')

import shutil
TARGET = '/content/drive/MyDrive/chakravyuh/analyzer_lora_colab'
shutil.copytree(OUTPUT_DIR, TARGET, dirs_exist_ok=True)
print('Saved to', TARGET)

## Next steps

1. **Scale**: on an A100 (RunPod, ~$2/hr), bump to `Qwen2.5-7B-Instruct` + 200 episodes (~$5).
2. **Push to HF Hub**: `huggingface-cli upload chakravyuh/analyzer-lora $OUTPUT_DIR`.
3. **Drop into demo**: `server/demo_ui.py` picks up the adapter automatically from `checkpoints/`.
4. **Run full frontier comparison**:
   ```
   python -m eval.frontier_baseline --providers groq gemini openai anthropic
   ```

Chakravyuh · Meta PyTorch OpenEnv Hackathon · April 2026